# DocRestore - Training on Google Colab

Run all cells top to bottom. Steps:
1. Mount Google Drive
2. Clone the repo
3. Install dependencies
4. Generate training data
5. Train DocRes and NAFNet
6. Save checkpoints to Drive

In [ ]:
# Step 1 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 - Clone the repo (skip if already cloned)
import os

REPO_DIR = '/content/doc-restore'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shamathmika/doc-restore.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repo already exists - pulled latest changes.')

%cd {REPO_DIR}

In [ ]:
# Step 3 - Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Step 4 - Generate training data
# Downloads ~50 arXiv PDFs and applies Augraphy degradations (~900 pairs)
# Takes 10-20 minutes. Skip this cell if data/train.csv already exists.
import os
if not os.path.exists('data/train.csv'):
    !python data/download_shabby.py
    !python data/split.py
else:
    print('Training data already exists - skipping.')

In [ ]:
# Step 5a - Train DocRes (50 epochs, ~1-2 hours on T4)
!python train/train_docres.py --config configs/docres.yaml

In [ ]:
# Step 5b - Train NAFNet (50 epochs, ~1-2 hours on T4)
!python train/train_nafnet.py --config configs/nafnet.yaml

In [ ]:
# Step 6 - Save checkpoints and loss logs to Google Drive
# Saved to MyDrive/doc-restore/checkpoints/
import sys
sys.path.insert(0, '/content/doc-restore')
from train.colab_utils import sync_checkpoints_to_drive
sync_checkpoints_to_drive()

In [ ]:
# (Optional) Run the Gradio demo with a public share link
!python demo/app.py --share